# InsPLAD Gold v2 Final — Reference Ablation

최종 `gold_v2_final` 246장에서 Reference 조건을 비교합니다.

- Image-only: 이미 끝난 `gold_v2_baseline` 결과 재사용
- Normal Reference: 기존 gold QA에 저장된 고정 정상 reference
- Positive Reference: supervised TRAIN anomalous bank에서 query GT를 사용하지 않고 DINOv2 cosine nearest-neighbor로 1장 검색
- Normal + Positive Reference: 두 reference를 동시에 제공

Positive retrieval은 target과 exact content duplicate인 train image를 제외합니다.
전체 결과는 Drive에 저장하고 핵심 결과는 runner 종료 시 GitHub `results/latest/gold_v2_reference/`에 자동 업로드합니다.


In [ ]:
# 0) 설정
REPO = 'arnold6444/lvlm_insplad'
REPO_REF = 'main'
QA_FOLDER = 'lvlm_insplad_qa/gold_v2_final'
BASELINE_FOLDER = 'lvlm_insplad_results/gold_baseline_v2'
RESULT_FOLDER = 'lvlm_insplad_results/gold_reference_v2'
REFERENCE_BUILD_FOLDER = 'lvlm_insplad_results/gold_reference_v2/positive_reference_build'

MODELS_TO_RUN = ['qwen3_5_4b','internvl3_5_8b','minicpm_v_4_5','llava_onevision_7b']
# Luxia까지 원하면 위 리스트 끝에 'luxia' 추가
CONDITIONS = ['normal_reference','positive_reference','normal_positive_reference']
CHECKPOINT_EVERY = 5
DINO_MODEL_ID = 'facebook/dinov2-small'
DINO_BATCH_SIZE = 24

SUP_ZIP_NAME = 'supervised_fault_classification.zip'
UNSUP_ZIP_NAME = 'unsupervised_anomaly_detection.zip'
EXPECTED_SUP_ZIP_SIZE = 874_753_167
EXPECTED_UNSUP_ZIP_SIZE = 1_170_967_719


In [ ]:
# 1) Drive mount + 최신 repo clone + 환경 준비
import base64, json, os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = Path('/content/drive/MyDrive')
QA_DIR = DRIVE_ROOT / QA_FOLDER
BASELINE_DIR = DRIVE_ROOT / BASELINE_FOLDER
RESULT_DIR = DRIVE_ROOT / RESULT_FOLDER
REFERENCE_BUILD_DIR = DRIVE_ROOT / REFERENCE_BUILD_FOLDER
RESULT_DIR.mkdir(parents=True, exist_ok=True)
REFERENCE_BUILD_DIR.mkdir(parents=True, exist_ok=True)

REPO_DIR = Path('/content/lvlm_insplad')
if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)

def run(cmd, cwd=None, env=None):
    cmd=[str(x) for x in cmd]
    print('+', ' '.join(cmd[:9]) + (' ...' if len(cmd)>9 else ''))
    subprocess.run(cmd, cwd=cwd, env=env, check=True)

github_token = userdata.get('GITHUB_TOKEN')
if not github_token: raise RuntimeError('GITHUB_TOKEN이 없습니다.')
os.environ['GITHUB_TOKEN'] = github_token
auth = base64.b64encode(f'x-access-token:{github_token}'.encode()).decode()
git_env=os.environ.copy(); git_env.update({'GIT_CONFIG_COUNT':'1','GIT_CONFIG_KEY_0':'http.extraHeader','GIT_CONFIG_VALUE_0':f'AUTHORIZATION: basic {auth}','GIT_TERMINAL_PROMPT':'0'})
run(['git','clone','--depth','1','--branch',REPO_REF,f'https://github.com/{REPO}.git',REPO_DIR], env=git_env)
del auth, git_env

for secret in ['HF_TOKEN','LUXIA_API_KEY']:
    try: value=userdata.get(secret)
    except Exception: value=None
    if value: os.environ[secret]=value

run([sys.executable,'-m','pip','install','-q','transformers==5.14.0','accelerate==1.14.0','sentencepiece==0.2.2','pillow','pandas','numpy','tqdm'])
print('repo:', subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO_DIR,text=True).strip())
print('QA:', QA_DIR)
print('results:', RESULT_DIR)


In [ ]:
# 2) QA / source ZIP / baseline 결과 확인
import torch

def find_file(name):
    direct=DRIVE_ROOT/name
    if direct.is_file(): return direct
    matches=list(DRIVE_ROOT.rglob(name))
    if not matches: raise FileNotFoundError(name)
    return matches[0]

if not QA_DIR.is_dir(): raise FileNotFoundError(f'QA folder 없음: {QA_DIR}')
SUP_ZIP=find_file(SUP_ZIP_NAME)
UNSUP_ZIP=find_file(UNSUP_ZIP_NAME)
if SUP_ZIP.stat().st_size != EXPECTED_SUP_ZIP_SIZE: raise RuntimeError('supervised ZIP size mismatch')
if UNSUP_ZIP.stat().st_size != EXPECTED_UNSUP_ZIP_SIZE: raise RuntimeError('unsupervised ZIP size mismatch')
BASELINE_LEADERBOARD=BASELINE_DIR/'leaderboard.csv'
if not BASELINE_LEADERBOARD.is_file():
    raise FileNotFoundError(f'baseline leaderboard 없음: {BASELINE_LEADERBOARD}')

sys.path.insert(0,str(REPO_DIR/'scripts'))
import run_gold_v2_baselines as v2_runner
rows=v2_runner.load_gold_qa_v2(QA_DIR)
from collections import Counter
print('QA counts:', Counter(r['question_type'] for r in rows))
print('total QA:', len(rows))
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음')
if not torch.cuda.is_available(): raise RuntimeError('GPU Runtime을 사용하세요.')


In [ ]:
# 3) Leakage-safe Positive Reference manifest 생성 — 이미 있으면 재사용
POSITIVE_MANIFEST = REFERENCE_BUILD_DIR/'positive_reference_manifest.csv'
if not POSITIVE_MANIFEST.is_file():
    run([sys.executable, REPO_DIR/'scripts/build_gold_v2_positive_references.py',
         '--qa-dir', QA_DIR, '--supervised-zip', SUP_ZIP, '--out-dir', REFERENCE_BUILD_DIR,
         '--model-id', DINO_MODEL_ID, '--batch-size', str(DINO_BATCH_SIZE)])
else:
    print('재사용:', POSITIVE_MANIFEST)

import pandas as pd
summary=pd.read_csv(REFERENCE_BUILD_DIR/'positive_reference_summary.csv')
display(summary)
manifest=pd.read_csv(POSITIVE_MANIFEST)
print('positive references:', len(manifest))
print('target GT used:', manifest['target_gt_used_for_selection'].astype(str).unique().tolist())
display(manifest.head())


In [ ]:
# 4) Reference ablation 실행 — 끊기면 같은 셀 재실행하면 prediction checkpoint 재사용
cmd=[sys.executable, REPO_DIR/'scripts/run_gold_v2_reference_ablation.py',
     '--qa-dir', QA_DIR, '--sup-zip', SUP_ZIP, '--unsup-zip', UNSUP_ZIP,
     '--positive-manifest', POSITIVE_MANIFEST, '--out-dir', RESULT_DIR,
     '--baseline-leaderboard', BASELINE_LEADERBOARD,
     '--checkpoint-every', str(CHECKPOINT_EVERY),
     '--conditions', *CONDITIONS, '--models', *MODELS_TO_RUN]
run(cmd, env=os.environ.copy())


In [ ]:
# 5) 결과 확인 — runner가 GitHub publish까지 자동 수행
import pandas as pd, json
leaderboard=pd.read_csv(RESULT_DIR/'leaderboard.csv')
comparison=pd.read_csv(RESULT_DIR/'ablation_comparison.csv')
deltas=pd.read_csv(RESULT_DIR/'ablation_vs_baseline.csv')
print('Reference leaderboard')
display(leaderboard.sort_values(['anomaly_f1','defect_macro_f1','localization_accuracy'],ascending=False))
print('Image-only 포함 전체 비교')
display(comparison)
print('Baseline 대비 변화')
display(deltas.sort_values(['model_key','anomaly_f1_delta'],ascending=[True,False]))
print((RESULT_DIR/'run_status.json').read_text(encoding='utf-8'))
print('GitHub: results/latest/gold_v2_reference/')


## 다음 단계

이 notebook이 끝나면 ChatGPT에 `reference 결과 나왔어 확인해봐`라고 하면 됩니다.
그다음에는 같은 gold_v2_final에서 Vision Expert(PatchCore / Dinomaly / AnomalyCLIP / ReConPatch 계열) screening으로 넘어갑니다.
